In [1]:
import duckdb
from pathlib import Path

# 1. Establish Directory Paths
BASE_DIR = Path.cwd().parent
RESULT_DIR = BASE_DIR / "results"

# 2. Initialize the Phase 2 OLAP Warehouse Database
db_path = RESULT_DIR / "step03_01_data_warehouse.duckdb"
con = duckdb.connect(str(db_path))
print(f"Connected to OLAP Warehouse: {db_path.name}")

# 3. Construct the Star Schema View dynamically reading from your CSVs
star_schema_query = f"""
CREATE OR REPLACE VIEW olap_star_schema AS

-- Step A: Construct the Central Fact Table (Merging Delay & Schedule)
WITH fact_delay AS (
    SELECT 
        d.date,
        d.train_no,
        d.station_no,
        d.station_name,
        d.delay,
        s.distance_from_origin,
        s.arrival_day,
        s.arrival_time,
        s.departure_day,
        s.departure_time,
        -- Dynamically resolve operational context based on sequence
        CASE WHEN d.station_no = 1 THEN 'Departure Delay' ELSE 'Arrival Delay' END AS delay_type
    FROM '{RESULT_DIR.as_posix()}/combined_delay_cleaned.csv' d
    JOIN '{RESULT_DIR.as_posix()}/combined_schedule_imputed.csv' s 
      ON d.train_no = s.train_no 
     AND d.station_no = s.station_no
)

-- Step B: Link to Dimension Tables (The Starburst Branches)
SELECT 
    -- Fact Measures & Degenerate Dimension
    f.date,
    f.train_no,
    f.station_name,
    f.station_no,
    f.distance_from_origin,
    f.arrival_day,
    f.arrival_time,
    f.departure_day,
    f.departure_time,
    f.delay AS delay_minutes,
    f.delay_type,
    
    -- Time Dimension (dim_date)
    dt.year,
    dt.quarter,
    dt.month,
    dt.month_name,
    dt.day_of_month,
    dt.day_name,
    dt.day_of_week,
    dt.is_weekend,
    
    -- Train Dimension (dim_train_resolved)
    tr.train_name,
    tr.primary_type_code,
    tr.is_premium_special,
    
    -- Station Dimension (dim_station_resolved)
    st.station_full_name,
    st.station_zone,
    st.station_address
    
FROM fact_delay f
JOIN '{RESULT_DIR.as_posix()}/dim_date.csv' dt 
  ON f.date = dt.date_id
JOIN '{RESULT_DIR.as_posix()}/dim_train_resolved.csv' tr 
  ON f.train_no = tr.train_no
JOIN '{RESULT_DIR.as_posix()}/dim_station_resolved.csv' st 
  ON f.station_name = st.station_name;
"""

print("Building Star Schema View over Phase 1 results...")
con.execute(star_schema_query)
print("-> olap_star_schema successfully created.")

# 4. Test the OLAP capabilities (Roll-up Aggregation)
test_cube = """
SELECT 
    quarter,
    primary_type_code,
    COUNT(delay_minutes) AS total_delay_events,
    ROUND(AVG(delay_minutes), 2) AS average_delay_minutes
FROM olap_star_schema
GROUP BY quarter, primary_type_code
ORDER BY quarter, total_delay_events DESC
LIMIT 10;
"""

print("\n--- OLAP CUBE TEST (ROLL-UP BY QUARTER & TRAIN TYPE) ---")
results = con.execute(test_cube).df()
print(results.to_string(index=False))



Connected to OLAP Warehouse: step03_01_data_warehouse.duckdb
Building Star Schema View over Phase 1 results...
-> olap_star_schema successfully created.

--- OLAP CUBE TEST (ROLL-UP BY QUARTER & TRAIN TYPE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 quarter primary_type_code  total_delay_events  average_delay_minutes
       1       PASS-TRAINS             4210568                  33.28
       1        EXP-TRAINS             3179570                  40.23
       1         SF-TRAINS             1186109                  49.01
       1        T18-TRAINS               85694                  16.02
       1        PRM-TRAINS               36730                 104.60
       1        RAJ-TRAINS               28282                  47.07
       1        SHT-TRAINS               24252                  16.34
       1        GRB-TRAINS               23829                  64.50
       2       PASS-TRAINS             4518548                  30.67
       2        EXP-TRAINS             3302774                  35.24
